In [ ]:
from fastai.vision.all import *
from torch.nn import *


def Zip(*a):
    return np.stack([np.array(a, "O") for a in a], 1)


def Flat(a):
    return a.ravel().tolist()


def Unzip(a, n):
    return np.array(a, "O").reshape(-1, n)


def Rand(a):
    return (np.random.rand(len(a)) * a).astype(int)


def Slice(a, i, j):
    return a.split(Flat(np.diff(Zip(i, j).ravel(), prepend=0, append=len(a))))[1::2]


class Dat:
    m = {}
    M = {}

    def __init__(s, a, b=8):
        x = []
        l = []
        y = []
        for a in open(f"../bert/albert/CLUEdataset/cmnli/cmnli_public/{a}.json"):
            a = loads(a)
            if a["label"] != "-":
                for i in "sentence1", "sentence2":
                    if b:
                        t = a[i]
                    else:
                        t = [a for a in a[i] if a in s.m]
                    x.extend(s.m.setdefault(t, len(s.m)) for t in t)
                    l.append(len(t))
                y.append(s.M.setdefault(a["label"], len(s.M)))
        s.x = torch.tensor(x, device="cuda")
        s.l = np.array(l)
        s.t = np.r_[0, s.l[:-1]].cumsum()
        s.y = torch.tensor(y, device="cuda")
        s.b = b

    def __len__(s):
        return len(s.y) // s.b if s.b else 1

    def __iter__(s):
        if s.b:
            for t in range(len(s)):
                t = np.random.choice(len(s.y), s.b, 0)
                t.sort()
                T = (t[:, None] * 2 + [0, 1]).ravel()
                a = s.l[T]
                l = 1 + Rand(a)
                i = s.t[T] + Rand(a - l)
                yield [torch.cat(Slice(s.x, i, i + l)), l], s.y[torch.tensor(t, device="cuda")]
        else:
            yield [s.x, s.l], s.y


class At(Sequential):
    def forward(s, x):
        return super().forward(x[0]), x[1]


class All(Sequential):
    def forward(s, x):
        return super().forward(x) if len(x) > 1 else x


class Red(Sequential):
    def forward(s, x):
        x, l = x
        while len(x) > len(l):
            t = Rand(l) & 4294967294
            l = Zip(t, l & 1, (l & 4294967294) - t)
            x = Unzip(x.split(Flat(l)), 3)
            l[:, ::2] >>= 1
            x[:, ::2] = Unzip(super().forward(torch.cat(Flat(x[:, ::2])).view(-1, x[0, 0].size(1) * 2)).split(Flat(l[:, ::2])), 2)
            x = torch.cat(Flat(x))
            l = l.sum(1)
        return x


class SymLinear(Linear):
    def forward(s, x):
        return super().forward(torch.stack([x.view(len(x) // 2, -1), x.view(len(x) // 2, 2, -1).fliplr().view(len(x) // 2, -1)], 1)).view(len(x), -1)


class Sum(Module):
    @staticmethod
    def forward(x):
        return x.view(len(x) // 2, 2, -1).sum(1)

In [ ]:
m = Learner(
    DataLoaders(Dat("train", 64), Dat("dev", 0)),
    Sequential(
        At(Embedding(len(Dat.m), 128)),
        Red(Linear(256, 128), All(BatchNorm1d(128, track_running_stats=0)), Swish()),
        SymLinear(256, 128, 0),
        BatchNorm1d(128, track_running_stats=0),
        Swish(),
        Sum(),
        Linear(128, 3),
    ).cuda(),
    CrossEntropyLoss(),
    metrics=accuracy,
    cbs=ShowGraphCallback,
)
m.lr_find()

In [ ]:
m.fit_one_cycle(50, max(_), 9, 99, .01)

In [ ]:
m.fit_one_cycle(100, 0.0001, 9, 99, .01)

In [ ]:
m.save('cmnli')